In [1]:
%pip install -r requirements.txt


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

PROJECT = Path.cwd()

# If your notebook is saved inside code/business_entity_resolution/,
# this points to ER_submission/dataset.
DATASET = (PROJECT / "../../dataset").resolve()

CACHE = PROJECT / "cache"
WORK = PROJECT / "work"
MODELS = PROJECT / "models"
OUTPUT = (PROJECT / "../../output").resolve()
LEXICON = PROJECT / "artifacts" / "translit_lexicon.json"

print("Project:", PROJECT)
print("Dataset:", DATASET)
print("Output:", OUTPUT)

assert (DATASET / "test" / "test_source1.tsv").exists(), "test_source1.tsv was not found."
assert (DATASET / "test" / "test_source2.tsv").exists(), "test_source2.tsv was not found."
assert (DATASET / "test" / "test_source3.tsv").exists(), "test_source3.tsv was not found."

os.environ["PYTHONPATH"] = str(PROJECT / "src")

command = [
    sys.executable, "src/predict.py",
    "--data", str(DATASET),
    "--split", "test",
    "--cache", str(CACHE),
    "--models", str(MODELS),
    "--lexicon", str(LEXICON),
    "--out", str(OUTPUT),
    "--work", str(WORK / "test"),
    "--shard-rows", "1500000",
]

subprocess.run(command, check=True)


Project: C:\Users\RAHUL\Downloads\er_code_package\code\business_entity_resolution
Dataset: C:\Users\RAHUL\Downloads\er_code_package\dataset
Output: C:\Users\RAHUL\Downloads\er_code_package\output


In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path("../../output").resolve()

matching_file = output_dir / "matching_results.tsv"
candidates_file = output_dir / "candidate_pairs.tsv"

print("matching_results exists:", matching_file.exists())
print("candidate_pairs exists:", candidates_file.exists())

matches = pd.read_csv(matching_file, sep="\t")
candidates = pd.read_csv(candidates_file, sep="\t")

print("\nmatching_results.tsv")
print(matches.shape)
display(matches.head())

print("\ncandidate_pairs.tsv")
print(candidates.shape)
display(candidates.head())


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

PROJECT = Path.cwd()
DATASET = (PROJECT / "../../dataset").resolve()
OUTPUT = (PROJECT / "../../output").resolve()
VALIDATOR = PROJECT / "utils" / "validate_submission.py"

assert VALIDATOR.exists(), f"Put the official validator here: {VALIDATOR}"

result = subprocess.run(
    [
        sys.executable,
        str(VALIDATOR),
        "--matching", str(OUTPUT / "matching_results.tsv"),
        "--candidate", str(OUTPUT / "candidate_pairs.tsv"),
        "--test-dir", str(DATASET / "test"),
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("Validator exit code:", result.returncode)


In [ ]:
import shutil
from pathlib import Path

ROOT = Path("../../").resolve()  # ER_submission/
OUTPUT = ROOT / "output"
CODE = ROOT / "code"
FINAL_ZIP = ROOT / "final_submission.zip"

assert (OUTPUT / "matching_results.tsv").exists()
assert (OUTPUT / "candidate_pairs.tsv").exists()
assert (CODE / "business_entity_resolution" / "Documentation_template.md").exists()

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

shutil.make_archive(
    base_name=str(FINAL_ZIP.with_suffix("")),
    format="zip",
    root_dir=str(ROOT),
    base_dir="output",
)

# Add code/ into the same ZIP.
import zipfile

with zipfile.ZipFile(FINAL_ZIP, "a", zipfile.ZIP_DEFLATED) as zf:
    for file in CODE.rglob("*"):
        if file.is_file() and "__pycache__" not in file.parts:
            zf.write(file, file.relative_to(ROOT))

print("Created:", FINAL_ZIP)
